# Guide rapide — Interface de communication SHM

Ce guide vous explique comment connecter votre programme Python à la mémoire partagée, lire et modifier les variables communes, puis fermer proprement votre connexion.

La communication fonctionne entre plusieurs programmes indépendants, sans serveur local. La synchronisation des accès est gérée par l'interface.


## 1. Organisation des dossiers

Conservez les trois fichiers de communication ensemble. Placez votre programme dans un dossier parallèle :

```text
content/
├── shm/
│   ├── shared_state.py
│   ├── launch_shm.py
│   └── client.py
│
├── camera/
│   └── camera.py
│
└── stage/
│   └── stage.py
│
└── interface/
    └── interface.py
```

- `shared_state.py` contient la mémoire partagée et les verrous.
- `launch_shm.py` crée et initialise la mémoire.
- `client.py` fournit des exemples de fonctions d'accès.

Ne déplacez pas les trois fichiers hors de `./shm`.


## 2. Importer l'interface depuis votre dossier

Dans le fichier `main.py` de votre programme, ajoutez ce bloc au début. Il permet à Python de trouver `shared_state.py` dans le dossier voisin `SHM_INTERFACE`, même si vous lancez votre programme depuis un autre répertoire.


In [ ]:
from pathlib import Path
import sys

# SECTION : SHM Interface Import

SHM_DIRECTORY = (Path(__file__).resolve().parent.parent / "shm")
sys.path.insert(0, str(SHM_DIRECTORY))

from shared_state import SharedState


Cette méthode suppose que `VOTRE_PROGRAMME` et `SHM_INTERFACE` sont bien au même niveau. Si votre arborescence diffère, adaptez le chemin.


## 3. Initialiser la mémoire (une seule fois)

Avant la première utilisation, créez la mémoire depuis le dossier `sources` :

```bash
python launch_shm.py
```

Les variables actuellement disponibles sont :

| Variable | Type | Valeur initiale |
|---|---|---|
| `targetLocation` | Texte | `""` |
| `currentLocation` | Texte | `""` |

Ne relancez pas l'initialisation à chaque démarrage de votre programme. Connectez-vous simplement à la mémoire existante.


## 4. Se connecter, lire et modifier

Après l'import, connectez-vous avec `SharedState()` :



In [ ]:
# SECTION : SHM Connection

shm = SharedState()

try:
    # Read shared variables.
    current = shm.read("currentLocation")
    target = shm.read("targetLocation")

    print("Current location:", current)
    print("Target location :", target)

    # Update a shared variable.
    shm.write("targetLocation", "Paris")

    # Read the updated value.
    print("New target:", shm.read("targetLocation"))

finally:
    # Close this program's connection.
    shm.close()


### Les trois opérations à connaître

```python
# Lire une variable
value = shm.read("currentLocation")

# Remplacer sa valeur
shm.write("targetLocation", "Paris")

# Lire, transformer et réécrire en une opération protégée
shm.modify("currentLocation", lambda old: "Lyon")
```

`modify()` est utile lorsque la nouvelle valeur dépend de l'ancienne : la lecture et l'écriture sont alors protégées ensemble.


## 5. Verrous : ce qui se passe automatiquement

Chaque variable possède son propre verrou :

- **Lecture (`LOCK_SH`)** : plusieurs programmes peuvent lire la même variable simultanément.
- **Écriture (`LOCK_EX`)** : un seul programme peut modifier la variable ; les autres lecteurs et rédacteurs attendent.

Par exemple, si un programme modifie `targetLocation`, un autre programme qui veut lire ou modifier `targetLocation` attend que l'écriture se termine. Il reprend ensuite automatiquement lorsque le verrou est libéré.

En revanche, un programme peut accéder à `currentLocation` pendant qu'un autre travaille sur `targetLocation` : les verrous sont indépendants.

`flock()` ne garantit pas un ordre FIFO entre les programmes en attente.


## 6. Règles à retenir

1. Utilisez toujours `read()`, `write()` et `modify()` : ne manipulez pas directement le contenu de la mémoire.
2. Respectez exactement les noms des variables (`targetLocation`, `currentLocation`).
3. Gardez les opérations protégées courtes pour ne pas faire attendre inutilement les autres programmes.
4. Appelez `shm.close()` à la fin de votre programme. Le bloc `try / finally` de l'exemple s'en charge même en cas d'erreur.
5. N'appelez pas `shm.destroy()` depuis votre programme : cette méthode supprime le segment partagé et doit être réservée à l'arrêt coordonné de l'ensemble des programmes.
6. Ne supprimez et ne remplacez pas les fichiers de verrouillage pendant que les programmes fonctionnent.

**Environnement :** cette implémentation utilise `fcntl.flock()` et est donc prévue pour Linux. Si `shared_state.py` utilise `track=False`, Python 3.13 ou une version ultérieure est nécessaire.
